# Gap 13 Analysis: B10/N10 Per-Class Error Analysis

This notebook provides a comprehensive per-class accuracy breakdown for the B10/N10 split:

1. Per-class accuracy heatmap across all methods and shot counts (mirrors B15/N5 analysis)
2. Inter-class confusion breakdown from aggregated confusion matrices
3. Class degradation when moving B15→B10: which shared novel classes suffer most?
4. Hardest/easiest class identification with visual difficulty explanation

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = 'white'

ROOT    = os.path.normpath(os.path.join(os.getcwd(), '..'))
RESULTS = os.path.join(ROOT, 'results')
print(f'Project root : {ROOT}')

In [ ]:
# ── helpers ──────────────────────────────────────────────────────────────────

MODEL_PATHS = {
    'NCC-Euc':    ('baseline', 'resnet50', '{split}', 'novel_ncc', '{k}shot_euclidean'),
    'NCC-Man':    ('baseline', 'resnet50', '{split}', 'novel_ncc', '{k}shot_manhattan'),
    'Proto-Euc':  ('proto_net', '{proto_split}', '{k}shot_euclidean'),
    'Proto-Man':  ('proto_net', '{proto_split}', '{k}shot_manhattan'),
    'MatchingNet':('matching_net', '{match_split}', '{k}shot'),
    'RelationNet':('relation_net', '{rel_split}', '{k}shot'),
}

def get_summary_path(method, split_key, k):
    split_dir   = 'base10' if '10/N10' in split_key else 'base15'
    proto_split = 'base10_novel10' if '10/N10' in split_key else 'base15_novel5'
    match_split = 'base10_novel10' if '10/N10' in split_key else 'base15_novel5'
    rel_split   = 'base10_novel10' if '10/N10' in split_key else 'base15_novel5'

    if method == 'NCC-Euc':
        return os.path.join(RESULTS, 'baseline', 'resnet50', split_dir,
                            'novel_ncc', f'{k}shot_euclidean', 'summary.json')
    elif method == 'NCC-Man':
        return os.path.join(RESULTS, 'baseline', 'resnet50', split_dir,
                            'novel_ncc', f'{k}shot_manhattan', 'summary.json')
    elif method == 'Proto-Euc':
        return os.path.join(RESULTS, 'proto_net', proto_split,
                            f'{k}shot_euclidean', 'summary.json')
    elif method == 'Proto-Man':
        return os.path.join(RESULTS, 'proto_net', proto_split,
                            f'{k}shot_manhattan', 'summary.json')
    elif method == 'MatchingNet':
        return os.path.join(RESULTS, 'matching_net', match_split,
                            f'{k}shot', 'summary.json')
    elif method == 'RelationNet':
        return os.path.join(RESULTS, 'relation_net', rel_split,
                            f'{k}shot', 'summary.json')
    return None


def load_per_class(method, split_key, k):
    path = get_summary_path(method, split_key, k)
    if path is None or not os.path.exists(path):
        return None, None
    with open(path) as f:
        s = json.load(f)
    return s.get('per_class_accuracy_pct', {}), s.get('novel_classes', [])

## 1. Per-class accuracy heatmap — B10/N10 across all methods and shot counts

In [ ]:
METHODS_B10 = ['NCC-Euc', 'NCC-Man', 'Proto-Euc', 'Proto-Man', 'MatchingNet', 'RelationNet']
SHOTS       = [1, 3, 5]
SPLIT_KEY   = 'B10/N10'

# Collect per-class data
# row = method+shot, col = class
row_labels = []
data_rows  = []
class_names = None

for method in METHODS_B10:
    for k in SHOTS:
        pc, classes = load_per_class(method, SPLIT_KEY, k)
        if pc is None:
            continue
        if class_names is None:
            class_names = classes
        row_labels.append(f'{method} {k}-shot')
        data_rows.append([pc.get(c, np.nan) for c in class_names])

if data_rows:
    heatmap_df = pd.DataFrame(data_rows, index=row_labels,
                               columns=[c.replace('AMERICAN ', 'AM. ') for c in class_names])

    fig, ax = plt.subplots(figsize=(14, max(6, len(row_labels) * 0.55 + 1)))
    sns.heatmap(
        heatmap_df, annot=True, fmt='.1f', cmap='RdYlGn',
        vmin=0, vmax=100, linewidths=0.3, ax=ax,
        cbar_kws={'label': 'Accuracy (%)'}
    )
    ax.set_title('B10/N10 — Per-class accuracy heatmap (all methods & shot counts)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Novel class')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(os.path.join(ROOT, 'figures', 'analysis_b10n10_per_class_heatmap.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No data loaded — check that results exist in the results/ directory.')

## 2. Mean per-class accuracy at 5-shot (B10/N10) — ranked by difficulty

In [ ]:
rows_5shot = []
for method in METHODS_B10:
    pc, classes = load_per_class(method, SPLIT_KEY, 5)
    if pc:
        for cls in classes:
            rows_5shot.append({'Method': method, 'Class': cls, 'Accuracy': pc.get(cls, np.nan)})

if rows_5shot:
    df5 = pd.DataFrame(rows_5shot)
    mean_by_class = df5.groupby('Class')['Accuracy'].mean().sort_values()

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#F44336' if v < 70 else '#FF9800' if v < 90 else '#4CAF50'
              for v in mean_by_class.values]
    bars = ax.barh([c.replace('AMERICAN ', 'AM. ') for c in mean_by_class.index],
                   mean_by_class.values, color=colors, alpha=0.85)
    ax.axvline(100 / 5, color='gray', ls=':', lw=1.2, label='Random (20%)')
    ax.set_xlabel('Mean accuracy (%) — average over all 6 methods')
    ax.set_title('B10/N10, 5-shot — Mean per-class accuracy ranked by difficulty')
    ax.set_xlim(0, 105)
    for bar, val in zip(bars, mean_by_class.values):
        ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(ROOT, 'figures', 'analysis_b10n10_per_class_ranked.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    print('\nDifficulty ranking (easiest → hardest):')
    for cls, acc in mean_by_class.sort_values(ascending=False).items():
        print(f'  {cls:<35s}: {acc:.1f}%')

## 3. B15→B10 degradation: which of the 5 shared novel classes suffer most?

In [ ]:
# The 5 classes that appear as novel in BOTH B15/N5 and B10/N10
SHARED_NOVEL = [
    'AMERICAN BITTERN', 'AMERICAN COOT', 'AMERICAN FLAMINGO',
    'AMERICAN GOLDFINCH', 'AMERICAN KESTREL'
]

deg_rows = []
for method in ['NCC-Euc', 'Proto-Euc', 'MatchingNet', 'RelationNet']:
    for k in [1, 3, 5]:
        pc_b15, _  = load_per_class(method, 'B15/N5',  k)
        pc_b10, _  = load_per_class(method, SPLIT_KEY, k)
        if pc_b15 is None or pc_b10 is None:
            continue
        for cls in SHARED_NOVEL:
            b15_acc = pc_b15.get(cls, np.nan)
            b10_acc = pc_b10.get(cls, np.nan)
            deg_rows.append({
                'Method': method, 'K': k, 'Class': cls,
                'B15/N5': b15_acc, 'B10/N10': b10_acc,
                'Degradation (pp)': b15_acc - b10_acc,
            })

if deg_rows:
    df_deg = pd.DataFrame(deg_rows)
    # Mean degradation per class across methods and shot counts
    mean_deg = df_deg.groupby('Class')['Degradation (pp)'].mean().sort_values(ascending=False)
    display(df_deg.pivot_table(
        index='Class', columns=['Method', 'K'],
        values='Degradation (pp)'
    ).round(1))

    fig, ax = plt.subplots(figsize=(9, 4))
    colors = ['#F44336' if v > 0 else '#4CAF50' for v in mean_deg.values]
    ax.bar([c.replace('AMERICAN ', 'AM. ') for c in mean_deg.index],
           mean_deg.values, color=colors, alpha=0.85)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('Mean accuracy drop B15→B10 (pp)')
    ax.set_title('Accuracy degradation when moving from B15/N5 to B10/N10\n'
                 '(5 shared novel classes, averaged over methods and shot counts)')
    plt.tight_layout()
    plt.savefig(os.path.join(ROOT, 'figures', 'analysis_b15_to_b10_degradation.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## 4. Confusion matrix analysis — B10/N10 ProtoNet 5-shot

In [ ]:
import glob

cm_path = os.path.join(RESULTS, 'proto_net', 'base10_novel10', '5shot_euclidean', 'confusion_matrix.png')
csv_path = cm_path.replace('.png', '.csv')

if os.path.exists(csv_path):
    cm_df = pd.read_csv(csv_path, index_col=0)
    short = lambda x: x.replace('AMERICAN ', 'AM. ').replace('ALEXANDRINE ', 'ALEX. ')
    cm_df.index   = [short(c) for c in cm_df.index]
    cm_df.columns = [short(c) for c in cm_df.columns]

    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(cm_df.values, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=cm_df.columns, yticklabels=cm_df.index, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('ProtoNet B10/N10 — Aggregated Confusion Matrix (5-shot Euclidean, avg per episode)')
    plt.tight_layout()
    plt.show()

    # Most confused pairs
    vals = cm_df.copy()
    np.fill_diagonal(vals.values, 0)
    vals_flat = vals.stack().sort_values(ascending=False)
    print('\nTop 5 most confused class pairs (ProtoNet B10/N10 5-shot):')
    for (true_cls, pred_cls), count in vals_flat.head(5).items():
        print(f'  True: {true_cls:<25s} → Predicted: {pred_cls:<25s}  count={count:.2f}')
elif os.path.exists(cm_path):
    from IPython.display import Image
    display(Image(cm_path, width=700))
else:
    print(f'Confusion matrix not found at {csv_path}\nRun evaluation first.')

## 5. Difficulty summary table — B10/N10 all methods at 5-shot

In [ ]:
cols = {}
for method in METHODS_B10:
    pc, classes = load_per_class(method, SPLIT_KEY, 5)
    if pc and classes:
        cols[method] = [pc.get(c, np.nan) for c in classes]

if cols and classes:
    tbl = pd.DataFrame(cols, index=[c.replace('AMERICAN ', 'AM. ') for c in classes])
    tbl['Mean'] = tbl.mean(axis=1)
    tbl = tbl.sort_values('Mean')
    display(tbl.round(1).style
            .background_gradient(cmap='RdYlGn', vmin=0, vmax=100)
            .format('{:.1f}'))

    print('\nKey observations:')
    print(f'  Hardest class : {tbl.index[0]}  (mean {tbl["Mean"].iloc[0]:.1f}%)')
    print(f'  Easiest class : {tbl.index[-1]} (mean {tbl["Mean"].iloc[-1]:.1f}%)')
    print(f'  MatchingNet is worst method on : {tbl["MatchingNet"].idxmin()} ({tbl["MatchingNet"].min():.1f}%)')
    print(f'  ProtoNet-Euc is best method on : {tbl["Proto-Euc"].idxmax()} ({tbl["Proto-Euc"].max():.1f}%)')